<a href="https://colab.research.google.com/github/1pawn0/GNNs-in-ComputerNetworks/blob/main/notebooks/caida_as_relationships_serial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset:
[https://catalog.caida.org/dataset/as_relationships_serial_2](https://catalog.caida.org/dataset/as_relationships_serial_2)



In [ ]:
# @title Install dependencies based on torch version

import os
import torch

TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:3])
CUDA = f"cu{torch.version.cuda.replace('.', '')[:3]}" if torch.version.cuda else "cpu"

os.environ["TORCH"] = TORCH
del TORCH
os.environ["CUDA"] = CUDA
del CUDA

%pip install -q pyg_lib torch_scatter torch_sparse torch_geometric -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html
%pip install -qU "torch-geometric-temporal[index]" "polars[style,graph,plot]" torch-geometric-signed-directed selectolax hvplot

import torch_geometric
import torch_geometric_temporal
import torch_geometric_signed_directed

print(torch.__version__)
print(torch_geometric.__version__)
print(torch_geometric_temporal.__version__)


In [ ]:
# @title dataset download via aria2c + extract via lbzip2

import os, re, shutil, subprocess, logging, time
from pathlib import Path
from urllib.parse import urljoin
from urllib.request import Request, urlopen, HTTPRedirectHandler, build_opener
from urllib.error import URLError, HTTPError
from concurrent.futures import ThreadPoolExecutor
from selectolax.parser import HTMLParser
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("caida_pipeline")

COMPRESSED_DIR = Path("compressed_data")
EXTRACTED_DIR = Path("data")
MIN_FREE_GB = 5
TIMEOUT = 15
RETRIES = 3


class HeadRequest(Request):
    def get_method(self):
        return "HEAD"


class HeadRedirectHandler(HTTPRedirectHandler):
    """Re-issues HEAD (not GET) after a redirect, since urllib defaults to GET."""

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if code in (301, 302, 303, 307, 308):
            return HeadRequest(newurl, headers=req.headers)
        return super().redirect_request(req, fp, code, msg, headers, newurl)


head_opener = build_opener(HeadRedirectHandler)


def fetch_with_retry(url, method_opener=None, timeout=TIMEOUT, retries=RETRIES):
    opener = method_opener or urlopen
    for attempt in range(1, retries + 1):
        try:
            req = url if isinstance(url, Request) else Request(url)
            return opener(req, timeout=timeout) if method_opener else urlopen(req, timeout=timeout)
        except (URLError, HTTPError) as e:
            if attempt == retries:
                raise
            time.sleep(2**attempt)


def ensure_tools():
    missing = [t for t in ("aria2c", "lbzip2") if not shutil.which(t)]
    if missing:
        log.info(f"Installing missing tools: {missing}")
        subprocess.run(["sudo", "apt-get", "-qq", "update"], check=True)
        subprocess.run(["sudo", "apt-get", "-qq", "install", "-y", "aria2", "lbzip2"], check=True)


def free_disk_gb(path="."):
    return shutil.disk_usage(path).free / (1024**3)


def list_files_recursive(url, year_pattern=r"^\d{4}/$", depth=1):
    try:
        with fetch_with_retry(url) as resp:
            html = resp.read().decode("utf-8", errors="ignore")
    except (URLError, HTTPError) as e:
        log.warning(f"Failed to scan {url}: {e}")
        return []

    tree = HTMLParser(html)
    files, subdirs = [], []

    for node in tree.css("a"):
        href = node.attributes.get("href")
        if not href or href.startswith(("/", "?", "..")):
            continue
        full_url = urljoin(url, href)
        if href.endswith("/"):
            if depth > 0 and re.match(year_pattern, href):
                subdirs.append(full_url)
        elif href.endswith((".bz2", ".txt")):
            files.append(full_url)

    with ThreadPoolExecutor(max_workers=4) as pool:
        for result in pool.map(lambda d: list_files_recursive(d, year_pattern, depth - 1), subdirs):
            files.extend(result)

    return sorted(set(files))


def head_size(link):
    try:
        req = HeadRequest(link)
        with fetch_with_retry(req, method_opener=head_opener.open) as resp:
            size = int(resp.headers.get("Content-Length", 0))
        return link, link.split("/")[-1], size
    except (URLError, HTTPError) as e:
        log.warning(f"HEAD failed for {link}: {e}")
        return link, link.split("/")[-1], 0


def run_aria2_dl(info, dest_dir: Path):
    url, filename, size = info
    dest = dest_dir / filename
    if dest.exists() and size > 0 and dest.stat().st_size == size:
        return dest

    cmd = [
        "aria2c",
        "-x",
        "16",
        "-s",
        "16",
        "-j",
        "16",
        "-c",
        "--max-tries=5",
        "--retry-wait=3",
        "--summary-interval=0",
        "-d",
        str(dest_dir),
        "-o",
        filename,
        url,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        log.warning(f"Download failed for {filename}: {result.stderr[-300:]}")
    return dest


def run_extraction(path: Path, dest_dir: Path, cores: int, delete_source=True):
    if path.suffix != ".bz2":
        return path
    out = dest_dir / path.stem
    if out.exists():
        return out
    try:
        subprocess.run(f"lbzip2 -n {cores} -dc {path} > {out}", shell=True, check=True)
        if delete_source:
            path.unlink(missing_ok=True)
    except subprocess.CalledProcessError as e:
        log.error(f"Extraction failed for {path}: {e}")
        out.unlink(missing_ok=True)
        return None
    return out


def pipeline(url, keep_compressed=False):
    COMPRESSED_DIR.mkdir(exist_ok=True)
    EXTRACTED_DIR.mkdir(exist_ok=True)
    ensure_tools()

    log.info(f"Scanning {url} recursively...")
    links = list_files_recursive(url)
    log.info(f"Found {len(links)} candidate files.")

    with ThreadPoolExecutor(max_workers=10) as pool:
        meta = list(tqdm(pool.map(head_size, links), total=len(links), desc="Metadata"))

    data_files = [m for m in meta if m[2] > 0 and m[1].endswith(".bz2")]
    data_files.sort(key=lambda x: x[2], reverse=True)
    total_gb = sum(m[2] for m in data_files) / (1024**3)
    log.info(f"{len(data_files)} files to fetch, ~{total_gb:.2f} GB compressed.")

    if free_disk_gb() - total_gb * 5 < MIN_FREE_GB:
        log.warning("Estimated decompressed size may exceed free disk space. Proceed with caution.")

    with ThreadPoolExecutor(max_workers=8) as pool:
        paths = list(
            tqdm(
                pool.map(lambda x: run_aria2_dl(x, COMPRESSED_DIR), data_files),
                total=len(data_files),
                desc="Downloading",
            )
        )

    cores = os.cpu_count() or 2
    for p in tqdm(paths, desc="Extracting"):
        if p and p.exists():
            run_extraction(p, EXTRACTED_DIR, cores, delete_source=not keep_compressed)

    if not keep_compressed:
        try:
            COMPRESSED_DIR.rmdir()  # only succeeds if empty
            log.info(f"Removed empty directory: {COMPRESSED_DIR}")
        except OSError:
            log.warning(f"{COMPRESSED_DIR} not empty, some files may have failed extraction — left in place.")

    log.info("Done.")


if __name__ == "__main__":
    pipeline("https://publicdata.caida.org/datasets/as-relationships/serial-2/")


In [ ]:
# @title read txt file and create the dataframe
import polars as pl
from pathlib import Path

source_enum = pl.Enum(["bgp", "mlp", "ark"])
rel_enum = pl.Enum(["provider->customer", "peer-peer"])

data_path = Path("/content/data")
# sorting ensures files are processed in chronological order
all_files = sorted([f for f in data_path.glob("*.as-rel2.txt")])

print(f"Found {len(all_files)} txt files in total.")


def process_file(file_path):
    date_str = file_path.name.split(".")[0]

    return (
        pl
        .read_csv(
            file_path,
            has_header=False,
            separator="|",
            comment_prefix="#",
            new_columns=["as1", "as2", "rel", "source"],
            schema_overrides={
                "as1": pl.UInt32,
                "as2": pl.UInt32,
                "rel": pl.Int8,
                "source": pl.Utf8,
            },
        )
        .with_columns([
            pl.lit(date_str).str.to_date("%Y%m%d").alias("date"),
            pl.col("source").str.split(",").cast(pl.List(source_enum)),
            pl
            .when(pl.col("rel") == -1)
            .then(pl.lit("provider->customer"))
            .when(pl.col("rel") == 0)
            .then(pl.lit("peer-peer"))
            .cast(rel_enum)
            .alias("rel_label"),
            (pl.col("rel") == -1).alias("is_p2c"),
            (pl.col("rel") == 0).alias("is_p2p"),
        ])
        .with_columns([
            pl.col("source").list.contains("bgp").alias("from_bgp"),
            pl.col("source").list.contains("mlp").alias("from_mlp"),
            pl.col("source").list.contains("ark").alias("from_ark"),
            (pl.col("date").dt.year() % 100).cast(pl.UInt8).alias("year"),
            pl.col("date").dt.month().cast(pl.UInt8).alias("month"),
        ])
        .select([
            "as1",
            "as2",
            "rel",
            "rel_label",
            "is_p2c",
            "is_p2p",
            "source",
            "from_bgp",
            "from_mlp",
            "from_ark",
            "year",
            "month",
            "date",
        ])
    )


if all_files:
    # Concatenate results; with_row_index adds 'index' as the first column
    df = pl.concat([process_file(f) for f in all_files], how="vertical").with_row_index()
    print(f"Memory usage: {df.estimated_size('gb'):.2f} GB")

    display(df.glimpse(return_type="self"))
    print(f"Shape: {df.shape}")

else:
    print("No files found.")


In [ ]:
# @title Save `df` as parquet
parquet_filepath = Path("/content/drive/MyDrive/Research/datasets/caida_as_relationships_serial2.parquet")
df.write_parquet(parquet_filepath, statistics="full")

In [ ]:
# @title Read parquet dataset
from pathlib import Path
import polars as pl

parquet_filepath = Path("/content/drive/MyDrive/Research/datasets/caida_as_relationships_serial2.parquet")
df = pl.read_parquet(parquet_filepath)
print(f"Memory usage: {df.estimated_size('gb'):.2f} GB")
display(df.glimpse(return_type="self"))
print(f"Shape: {df.shape}")

### Exploratory Data Analysis

In [ ]:
# @title Value counts
import polars as pl
import hvplot.polars

hvplot.extension("matplotlib")
for col_name in df.drop("index", "date", "source").columns:
    print(f"----------------------  {col_name}  ----------------------")

    stats_df = (
        df[col_name]
        .value_counts(
            name="percentage",
            normalize=True,
            sort=True,
        )
        .head(12)
        .with_columns([
            pl.col(col_name).cast(pl.String),
            pl.col("percentage").mul(100).round(4),
        ])
    )
    plot = stats_df.hvplot.bar(x=col_name, y="percentage", width=1200, height=600)
    display(plot)
    print("\n\n\n\n")
